# 05_dim_time

DML: dim_time — Time dimension, PK: play_timestamp.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

src = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.fct_plays")
    .filter(F.col("ingestion_date") == ingestion_date)
    .filter(F.col("run_id") == run_id)
    .select("played_at")
    .dropDuplicates(["played_at"])
    .withColumn("play_date",    F.to_date("played_at"))
    .withColumn("play_hour",    F.hour("played_at"))
    .withColumn("day_of_week",  F.dayofweek("played_at"))
    .withColumn("day_name",     F.date_format("played_at", "EEEE"))
    .withColumn("week_of_year", F.weekofyear("played_at"))
    .withColumn("month",        F.month("played_at"))
    .withColumn("month_name",   F.date_format("played_at", "MMMM"))
    .withColumn("year",         F.year("played_at"))
    .withColumn("is_weekend",   F.dayofweek("played_at").isin(1, 7))
)

upsert_delta(src, f"{CATALOG}.{SILVER_SCHEMA}.dim_time", ["played_at"])
print(f"dim_time: {src.count()} rows upserted")